<h1 style="
    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
    font-size: 36px;
    color: #2c3e50;
    background-color: #ecf0f1;
    padding: 20px;
    border-radius: 12px;
    text-align: center;
    box-shadow: 0px 4px 10px rgba(0, 0, 0, 0.1);">
    NextGen Calibration
</h1>

**Authors:** 

<ul style="line-height:1.5;">
<li>Ayman Nassar <a href="mailto:ayman.nassar@usu.edu">(ayman.nassar@usu.edu)</a></li>
<li>Joshua Cunningham <a href="mailto:jcunningham8@ua.edu">(jcunningham8@ua.edu)</a></li>
<li>David Tarboton <a href="mailto:david.tarboton@usu.edu">(david.tarboton@usu.edu)</a></li>
</ul>

**Last Updated:** 05/15/2026

**Purpose:**

This notebook is designed to automate the calibration workflow for NextGen streamflow. It focuses on calibrating key parameters of the **NOAH-OWP** and **Conceptual Functional Equivalent (CFE)** hydrologic models to improve the agreement between simulated and observed streamflow.

**Audience:**

Researchers and graduate students who are familiar with Jupyter Notebooks, basic Python, calibration, and basic hydrologic data analysis.

**Description:**

The calibration process uses [SPOTPY (https://github.com/thouska/spotpy)](https://github.com/thouska/spotpy) to optimize model parameters and improve the match between simulated and observed streamflow. In this workflow, you can select either the **DDS** (Dynamically Dimensioned Search) or **SCE-UA** (Shuffled Complex Evolution) algorithm to search for the best parameter values. You can also use **Kling–Gupta Efficiency (KGE)** or **Root Mean Square Error (RMSE)** as the performance measure to evaluate how well the model reproduces observed flow.

The workflow starts by defining the **simulation period**, the **gage ID**, and the **feature ID**, then downloading observed streamflow data from the [U.S. Geological Survey (USGS)](https://waterdata.usgs.gov/). SPOTPY then begins the calibration by selecting an initial set of parameters for the **NOAH-OWP** and **Conceptual Functional Equivalent (CFE)** models from predefined parameter ranges. These parameters are written into the NextGen configuration, and the model is run to generate simulated streamflow.

The simulated results are then compared with the observed data using the chosen performance measure (**KGE** or **RMSE**). Based on how well the model performs, the optimization algorithm adjusts the parameters and runs the model again. This cycle of **updating parameters**, **running the model**, and **evaluating performance** continues over many iterations. Gradually, the parameters are improved, resulting in a final calibrated model that provides the best match between simulated and observed streamflow for the selected gage.


**Data Description:**

The calibration workflow starts by working with **observed streamflow data** from the [U.S. Geological Survey (USGS)](https://waterdata.usgs.gov/). This data is collected at a specific location, identified by a **USGS gage ID**. The raw data is downloaded at 15-minute intervals using the USGS Instantaneous Values (IV) service. To make it match the model time step, the data is converted from cubic feet per second (cfs) to **cubic meters per second (m³/s)** and averaged into **hourly values**. This creates a clean time series of observed streamflow for the selected gage, which is then used as the reference for calibration and model performance checks.

Next, during each calibration iteration, the **model generates simulated streamflow** using a set of **adjustable parameters**. These parameters are updated in every iteration as part of the optimization process. In addition to the adjustable parameters, the model also uses several fixed input datasets that were prepared during the initial setup. These include the **hydrofabric** (describing the watershed and stream network), **AORC forcing data** (providing gridded meteorological variables such as precipitation, temperature, and radiation), and the **model configuration and realization files**.  

For each iteration, the model combines these fixed inputs with the new (updated) parameter values to run a simulation and produce a new streamflow time series. This simulated streamflow is then compared with the observed data to evaluate how well the model is performing. This comparison is repeated through many iterations until the parameter set that gives the best match between simulated and observed flow is found.

**Software Requirements:**
This notebook requires the following libraries:
> json: 2.0.9  
> datetime: 3.11.13  
> pathlib: 3.11.13

It also uses code from `cal_utils.py`

<div style="
    padding: 15px 20px; 
    background-color: #e2f0fe; 
    border-left: 6px solid #3b82f6; 
    color: #1e3a8a; 
    border-radius: 4px; 
    margin-bottom: 20px;
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif;
">
    <h3 style="margin-top: 0; color: #1e3a8a; font-weight: 700; display: flex; align-items: center; gap: 8px;">
        💡 Quick Note Before You Begin
    </h3>
    <p style="margin-bottom: 10px; font-size: 1.05em;">
        To make sure everything runs smoothly and all settings initialize correctly, <strong>please take a moment to restart the kernel before running the cells below.</strong>
    </p>
    <p style="margin: 0; font-size: 0.95em;">
        <strong>How to do this:</strong> Simply navigate to the <strong>Kernel</strong> menu at the top and select <span style="background-color: rgba(0,0,0,0.05); padding: 2px 6px; border-radius: 3px; border: 1px solid rgba(0,0,0,0.1);"><strong>“Restart Kernel and Clear Outputs of All Cells”</strong></span>. Thank you!
    </p>
</div>

<div style="background:#13294b; border-left:6px solid #5cd6ff; padding:12px 16px; border-radius:8px; margin-top:0; margin-bottom:0;">
  <h3 style="margin:0; font-size:20px; font-weight:700; color:#eaf7ff;">
    1. Prepare the Python Environment
  </h3>
</div>
Before running the calibration workflow, import all required Python libraries and modules.  
This notebook relies on the custom module <strong>cal_utils</strong>, which provides the core functionality for automated NextGen streamflow calibration using the SPOTPY optimization framework.

The <strong>cal_utils</strong> module includes:
- Tools for updating CFE and Noah-OWP parameters inside the <code>realization.json</code> file.
- Functions to execute the NextGen model using PyNGIAB within the containerized environment.
- Utilities to extract simulated streamflow results from T-route output files.
- Performance evaluation routines (e.g., KGE, RMSE) for comparing simulated results with USGS observations.

Together, these components enable an iterative, fully automated calibration workflow for NextGen hydrologic modeling.


In [ ]:
# ----------------------------- Importing Required Libraries -----------------------------
import json
import pandas as pd
from pathlib import Path
from datetime import datetime
from cal_utils import process_usgs_streamflow, run_spotpy, get_troute_output_name, update_model_params_from_csv

<div style="background:#13294b; border-left:6px solid #5cd6ff; padding:12px 16px; border-radius:8px; margin-top:24px; margin-bottom:0;">
  <h3 style="margin:0; font-size:20px; font-weight:700; color:#eaf7ff;">
    2. Set Inputs
  </h3>
</div>
<p style="margin-top:12px; font-size:15px; line-height:1.55;">
In this step, we define the key input variables required for the calibration workflow, including the USGS gage ID, the corresponding NextGen feature ID, the simulation period, and essential file paths. These inputs establish the link between the observed streamflow data, the model configuration, and the directory structure used for storing outputs. Setting these parameters ensures that the calibration workflow is correctly aligned with the target watershed, dataset, and model setup.
</p>

In [ ]:
gage_id = "10109001"  # USGS gage ID used to identify the streamflow observation site
feature_id = 2861391  # Unique feature ID in the NextGen routing network corresponding to the gage

start_date = "2017-10-01"       # Start date for the model simulation period
end_date = "2021-09-30"         # End date for the model simulation period
training_start_date = "2020-10-01"  # Start date for the calibration (training) period

data_root = "/home/jovyan/ngiab_preprocess_output"  # Root directory where model inputs/outputs are stored

realization_path = f"{data_root}/gage-{gage_id}/config/realization.json"  # Path to the model configuration file
observed_flow_path = f"{data_root}/{gage_id}_observed_flow.pkl"           # Path to the processed observed flow data
troute_output_path = (
    f"{data_root}/gage-{gage_id}/outputs/troute/{get_troute_output_name(realization_path)}"
)  # Path to the simulated t-route output file
data_dir = f"{data_root}/gage-{gage_id}"  # Base directory for all files related to the selected gage

# If observed flow data doesn't exist, retrieve it from USGS and save as a pickle file
if not Path(observed_flow_path).exists():
    process_usgs_streamflow(gage_id, start_date, end_date, output_path=observed_flow_path)


<div style="background:#13294b; border-left:6px solid #5cd6ff; padding:12px 16px; border-radius:8px; margin-top:24px; margin-bottom:0;">
  <h3 style="margin:0; font-size:20px; font-weight:700; color:#eaf7ff;">
    3. Run Calibration
  </h3>
</div>
<p style="margin-top:12px; font-size:15px; line-height:1.55;">
In this step, the calibration workflow is executed using the <code>run_spotpy</code> function. This function applies the selected optimization algorithm and objective function to systematically adjust model parameters, run the NextGen simulation, and evaluate performance against observed streamflow. Through repeated iterations, the routine identifies the parameter set that provides the best fit to the data. These optimized <strong>best_params</strong> values will then be used to generate the final calibrated model simulation.
</p>

### User Input Description

<table style="border-collapse: collapse; width: 100%; font-size: 14px;">
  <thead>
    <tr style="background-color: navy; color: white;">
      <th style="padding: 8px; text-align: left;">Parameter</th>
      <th style="padding: 8px; text-align: left;">Description</th>
      <th style="padding: 8px; text-align: left;">Example Value</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="padding: 8px; text-align: left;"><code>gage_id</code></td>
      <td style="padding: 8px; text-align: left;">USGS gage ID used to identify the streamflow location for calibration.</td>
      <td style="padding: 8px; text-align: left;"><code>"10109001"</code></td>
    </tr>
    <tr>
      <td style="padding: 8px; text-align: left;"><code>start_date</code></td>
      <td style="padding: 8px; text-align: left;">Start date of the overall model simulation period.</td>
      <td style="padding: 8px; text-align: left;"><code>"2015-10-01"</code></td>
    </tr>
    <tr>
      <td style="padding: 8px; text-align: left;"><code>end_date</code></td>
      <td style="padding: 8px; text-align: left;">End date of the overall model simulation period.</td>
      <td style="padding: 8px; text-align: left;"><code>"2022-10-01"</code></td>
    </tr>
    <tr>
      <td style="padding: 8px; text-align: left;"><code>training_start_date</code></td>
      <td style="padding: 8px; text-align: left;">Start date for the calibration (training) period used to evaluate model performance.</td>
      <td style="padding: 8px; text-align: left;"><code>"2018-10-01"</code></td>
    </tr>
    <tr>
      <td style="padding: 8px; text-align: left;"><code>observed_flow_path</code></td>
      <td style="padding: 8px; text-align: left;">Path to the processed observed streamflow data file.</td>
      <td style="padding: 8px; text-align: left;"><code>"/path/to/observed.pkl"</code></td>
    </tr>
    <tr>
      <td style="padding: 8px; text-align: left;"><code>troute_output_path</code></td>
      <td style="padding: 8px; text-align: left;">Path where the simulated t-route output will be stored.</td>
      <td style="padding: 8px; text-align: left;"><code>"/path/to/troute.nc"</code></td>
    </tr>
    <tr>
      <td style="padding: 8px; text-align: left;"><code>data_dir</code></td>
      <td style="padding: 8px; text-align: left;">Base directory containing all model input and output files.</td>
      <td style="padding: 8px; text-align: left;"><code>"/home/user/data"</code></td>
    </tr>
    <tr>
      <td style="padding: 8px; text-align: left;"><code>feature_id</code></td>
      <td style="padding: 8px; text-align: left;">Feature ID in the NextGen routing network corresponding to the gage.</td>
      <td style="padding: 8px; text-align: left;"><code>2861391</code></td>
    </tr>
    <tr>
      <td style="padding: 8px; text-align: left;"><code>algorithm</code></td>
      <td style="padding: 8px; text-align: left;">Optimization method used for calibration. Options include <code>"DDS"</code> or <code>"SCE"</code>.</td>
      <td style="padding: 8px; text-align: left;"><code>"DDS"</code></td>
    </tr>
    <tr>
      <td style="padding: 8px; text-align: left;"><code>objective_function</code></td>
      <td style="padding: 8px; text-align: left;">Performance metric guiding calibration, such as <code>"KGE"</code> or <code>"RMSE"</code>.</td>
      <td style="padding: 8px; text-align: left;"><code>"KGE"</code></td>
    </tr>
    <tr>
      <td style="padding: 8px; text-align: left;"><code>repetitions</code></td>
      <td style="padding: 8px; text-align: left;">Number of parameter sampling iterations during the calibration process. The number of repetitions should generally be at least greater than the number of calibration parameters.</td>
      <td style="padding: 8px; text-align: left;"><code>200</code></td>
    </tr>
    <tr>
      <td style="padding: 8px; text-align: left;"><code>dds_trials</code></td>
      <td style="padding: 8px; text-align: left;">Number of DDS trials (if using the DDS algorithm).</td>
      <td style="padding: 8px; text-align: left;"><code>1</code></td>
    </tr>
  </tbody>
</table>


<div style="border-left: 6px solid #1f77b4; background-color: #f4f8fb; padding: 15px; border-radius: 6px;">

## Calibration Outputs During Runtime

While the calibration process is running, the following folders are continuously updated inside:

`~/ngiab_preprocess_output/{hydrofabric_id}/calibration`

### 📊 `plots/`
Contains plots comparing the simulated streamflow from each calibration iteration against the observed streamflow.

### 📁 `iterations/`
Contains a CSV file summarizing the calibration results for all iterations completed so far.

The CSV file includes:

- Iteration number
- Objective-function value
- Calibrated parameter values for each iteration

The iteration with the best objective-function value is highlighted for easier identification.

</div>

In [ ]:
# For one iteration, a single-year calibration takes approximately 5–7 minutes.

best_params = run_spotpy(
    gage_id,
    start_date,
    end_date,
    training_start_date,
    observed_flow_path,
    troute_output_path,
    data_dir,
    feature_id,
    algorithm="DDS",
    objective_function="KGE",
    repetitions=6,
    dds_trials=1,
)

<div style="border-left: 6px solid #2ca02c; background-color: #f3fbf4; padding: 15px; border-radius: 6px;">

## Calibration Completed

Your calibration process is complete.

You can now select the best parameter values based on the objective-function results stored in:

`~/ngiab_preprocess_output/{hydrofabric_id}/calibration/iterations`

The best-performing iteration is identified in the CSV file by the value:

`is_best = True`

After selecting the desired parameter values, update them in:

`~/ngiab_preprocess_output/{hydrofabric_id}/config/realization.json`

These updated parameter values can then be used in future model simulations and evaluations. The following cell automatically updates the calibrated parameter values for subsequent model runs.

</div>

<div style="background:#13294b; border-left:6px solid #5cd6ff; padding:12px 16px; border-radius:8px; margin-top:24px; margin-bottom:0;">
  <h3 style="margin:0; font-size:20px; font-weight:700; color:#eaf7ff;">
    4. Apply Best Calibrated Parameters to <code>realization.json</code>
  </h3>
</div>

<p style="margin-top:12px; font-size:15px; line-height:1.55;">
This section updates the <code>realization.json</code> configuration file using the best calibrated parameter values obtained during the calibration. After the parameters are updated, you can proceed to the <strong>NextGen_Run</strong> Jupyter notebook to execute the simulation using the Python wrapper <strong>PyNGIAB</strong> with the newly calibrated parameter set.
</p>

In [ ]:
# Paths

# realization configuration file (realization.json) 
realization_path = Path(f"{data_root}/gage-{gage_id}/config/realization.json") # path of realization.josn file

# Calibration history: parameter values and metrics for each iteration; row with is_best == True is the best run
calibration_iterations_path = Path(
    f"{data_root}/gage-{gage_id}/calibration/iterations/calibration_iterations.csv"  # path of csv file in the calibration/iterations folder
)

# How CSV column names link to names inside realization.json
cfe_csv_to_json = {
    "b": "soil_params_b",
    "satpsi": "satpsi",
    "satdk": "satdk",
    "maxsmc": "maxsmc",
    "refkdt": "refkdt",
    "expon": "expon",
    "slope": "slope",
    "max_gw_storage": "max_gw_storage",
    "Kn": "K_nash_subsurface",
    "Klf": "K_lf",
    "Cgw": "Cgw",
}

noah_json_to_csv = {
    "MFSNO": "MFSNO",
    "MP": "MP",
    "RSURF_EXP": "RSURF_EXP",
    "CWP": "CWP",
    "VCMX25": "VCMX25",
    "RSURF_SNOW": "RSURF_SNOW",
    "SCAMAX": "SCAMAX",
}


# Step 1: Best calibration row
calibration_df = pd.read_csv(calibration_iterations_path)
best_row = calibration_df[calibration_df["is_best"] == True]

if best_row.empty:
    raise ValueError("No row with is_best == True in calibration_iterations.csv")

best_row = best_row.iloc[0]
print(f"Best iteration: {int(best_row['iteration'])}")

# Step 2: Load realization.json
with open(realization_path, "r") as file:
    realization = json.load(file)

model_modules = realization["global"]["formulations"][0]["params"]["modules"]

# Step 3: Update CFE and NoahOWP only where parameters already exist
total_updated = 0
for module in model_modules:
    model_type = module["params"].get("model_type_name", "")

    if model_type == "CFE":
        total_updated += update_model_params_from_csv(
            module, cfe_csv_to_json, best_row, "CFE"
        )
    elif model_type == "NoahOWP":
        total_updated += update_model_params_from_csv(
            module, noah_json_to_csv, best_row, "NoahOWP"
        )

# Step 4: Save
with open(realization_path, "w") as file:
    json.dump(realization, file, indent=4)

print(f"\nSaved {total_updated} total update(s) to:\n  {realization_path}")